# Bài 3.1: Fine-tuning `PhoBERT-base-v2` Phân tích Cảm xúc trên Tập `UIT-VSFC`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vietnamese-nlp-mastery/vietnamese-nlp-mastery/blob/main/03_Pretrained_Transformers/01_phobert_text_classification_uit_vsfc.ipynb)

> **Mục tiêu bài học:**
> 1. Hiểu kiến trúc và đặc thù Tokenizer của **`vinai/phobert-base-v2`** (giới hạn `max_position_embeddings = 258`, tương đương `max_length = 256`).
> 2. Thiết lập quy trình tiền xử lý chuẩn cho PhoBERT: **Unicode NFC + Dấu thanh kiểu cũ + Tách từ nối gạch dưới (`_`)**.
> 3. Huấn luyện hoàn chỉnh với Hugging Face `Trainer` và đánh giá bằng **Accuracy** & **Macro-F1** trên tập dữ liệu `UIT-VSFC`.

In [1]:
# Pipeline chuẩn Fine-tuning vinai/phobert-base-v2 trên UIT-VSFC
MODEL_NAME = "vinai/phobert-base-v2"
MAX_LENGTH = 256  # Lưu ý: PhoBERT chỉ hỗ trợ tối đa 256 subword tokens!

LABEL2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print(f"Model: {MODEL_NAME} | Max Length: {MAX_LENGTH} | Labels: {LABEL2ID}")

Model: vinai/phobert-base-v2 | Max Length: 256 | Labels: {'Negative': 0, 'Neutral': 1, 'Positive': 2}


In [2]:
# Code huấn luyện hoàn chỉnh với Hugging Face Transformers & Datasets
training_script_example = '''
import numpy as np
import evaluate
from datasets import load_dataset
from pyvi import ViTokenizer  # Hoặc dùng py_vncorenlp để đạt độ chính xác cao nhất
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# 1. Tải dữ liệu UIT-VSFC và Tokenizer của PhoBERT-base-v2
dataset = load_dataset("ura-hcmut/UIT-VSFC")
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

def preprocess_function(examples):
    # BẮT BUỘC: Tách từ tiếng Việt (nối bằng '_') trước khi đưa vào PhoBERT tokenizer
    segmented_texts = [ViTokenizer.tokenize(text) for text in examples["sentence"]]
    return tokenizer(segmented_texts, truncation=True, max_length=256)

tokenized_ds = dataset.map(preprocess_function, batched=True)

# 2. Khởi tạo mô hình PhoBERT cho bài toán phân loại 3 lớp cảm xúc
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2",
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

# 3. Định nghĩa hàm đánh giá Macro-F1 và Accuracy
f1_metric = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

# 4. Cấu hình TrainingArguments và chạy Trainer
training_args = TrainingArguments(
    output_dir="./outputs/phobert_uit_vsfc",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
'''
print("Đã sẵn sàng pipeline huấn luyện PhoBERT-base-v2 (Macro-F1 kỳ vọng trên UIT-VSFC: ~93.8%).")

Đã sẵn sàng pipeline huấn luyện PhoBERT-base-v2 (Macro-F1 kỳ vọng trên UIT-VSFC: ~93.8%).
